# Number Plate Detection: YOLO26m (+ Optional PaddleOCR) on CCPD2019

Notebook-first, real-data ALPR workflow with verifiable dataset checks, detector evaluation, optional OCR, and saved artifacts.

## 1. Problem Type and Method Fit

Task family: number plate detection (+ OCR optional).

- YOLO26m is used for robust plate localization.
- PaddleOCR is optional and runs on detected crops only.
- Detection metrics are reported quantitatively; OCR is reported honestly based on available text supervision.

## 2. Dataset Source and License Notes

Dataset: https://www.kaggle.com/datasets/hrishirajdevsarmah/ccpd2019

Use the dataset according to Kaggle terms and any source-license notes on the dataset page.

## 3. Environment and Reproducibility

In [1]:
import platform
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Python: 3.13.12
PyTorch: 2.6.0+cu124
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 4. Install and Verify Dependencies

In [2]:
import importlib
import subprocess
import sys

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

ensure_package('ultralytics')
ensure_package('kagglehub')
ensure_package('cv2', 'opencv-python')
ensure_package('matplotlib')
ensure_package('pandas')
ensure_package('yaml', 'pyyaml')
ensure_package('paddleocr')
print('Dependencies are available.')

e:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(
e:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


Dependencies are available.


## 5. Imports and Project Paths

In [3]:
import json
import os
import shutil
from pathlib import Path

import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from ultralytics import YOLO
from paddleocr import PaddleOCR

PROJECT_DIR = Path.cwd()
WORK_DIR = PROJECT_DIR / 'working'
DATA_DIR = WORK_DIR / 'data'
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
RUNS_DIR = PROJECT_DIR / 'runs'

WORK_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET = 'hrishirajdevsarmah/ccpd2019'
print(f'Project dir: {PROJECT_DIR}')

Project dir: e:\Github\Machine-Learning-Projects\Computer Vision\Number Plate Reader Pro\Source Code


## 6. Real Dataset Download (CCPD2019 from Kaggle)

In [ ]:
def check_kaggle_credentials():
    has_env = bool(os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))
    has_file = (Path.home() / '.kaggle' / 'kaggle.json').exists()
    if not has_env and not has_file:
        raise RuntimeError('Missing Kaggle credentials. Set KAGGLE_USERNAME/KAGGLE_KEY or create ~/.kaggle/kaggle.json')

def download_ccpd2019():
    check_kaggle_credentials()
    import kagglehub
    return Path(kagglehub.dataset_download(KAGGLE_DATASET))

raw_root = Path(download_ccpd2019())
print(f'Raw dataset path: {raw_root}')

## 7. Detect YOLO Dataset Root and Validate Splits

In [ ]:
def find_yolo_dataset_root(root: Path) -> Path:
    direct_yaml = root / 'data.yaml'
    if direct_yaml.exists():
        return root
    for child in root.iterdir():
        if child.is_dir() and (child / 'data.yaml').exists():
            return child
    raise RuntimeError('Could not locate YOLO data.yaml under downloaded CCPD2019 directory.')

YOLO_ROOT = find_yolo_dataset_root(raw_root)
DATA_YAML = YOLO_ROOT / 'data.yaml'

with DATA_YAML.open('r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

def resolve_split_path(split_key):
    split_value = data_cfg.get(split_key)
    if split_value is None:
        return None
    split_path = Path(split_value)
    if not split_path.is_absolute():
        split_path = YOLO_ROOT / split_path
    return split_path.resolve()

TRAIN_IMAGES = resolve_split_path('train')
VAL_IMAGES = resolve_split_path('val') if data_cfg.get('val') is not None else resolve_split_path('valid')
TEST_IMAGES = resolve_split_path('test')

if TRAIN_IMAGES is None or VAL_IMAGES is None:
    raise RuntimeError('Train/val split paths are missing in data.yaml')

TRAIN_LABELS = TRAIN_IMAGES.parent / 'labels'
VAL_LABELS = VAL_IMAGES.parent / 'labels'
TEST_LABELS = TEST_IMAGES.parent / 'labels' if TEST_IMAGES is not None else None

if not TRAIN_IMAGES.exists() or not TRAIN_LABELS.exists():
    raise RuntimeError('Train image/label folders not found')
if not VAL_IMAGES.exists() or not VAL_LABELS.exists():
    raise RuntimeError('Validation image/label folders not found')

train_images = sorted([p for p in TRAIN_IMAGES.iterdir() if p.is_file()])
train_labels = sorted([p for p in TRAIN_LABELS.iterdir() if p.is_file()])
val_images = sorted([p for p in VAL_IMAGES.iterdir() if p.is_file()])
val_labels = sorted([p for p in VAL_LABELS.iterdir() if p.is_file()])

if len(train_images) == 0:
    raise RuntimeError('No train images found')
if len(val_images) == 0:
    raise RuntimeError('No val images found')
if len(train_images) != len(train_labels):
    raise RuntimeError(f'Train mismatch: {len(train_images)} images vs {len(train_labels)} labels')
if len(val_images) != len(val_labels):
    raise RuntimeError(f'Val mismatch: {len(val_images)} images vs {len(val_labels)} labels')

if TEST_IMAGES is not None and TEST_IMAGES.exists() and TEST_LABELS is not None and TEST_LABELS.exists():
    test_images = sorted([p for p in TEST_IMAGES.iterdir() if p.is_file()])
    test_labels = sorted([p for p in TEST_LABELS.iterdir() if p.is_file()])
    print(f'Test split: images={len(test_images)}, labels={len(test_labels)}')

print(f'YOLO root: {YOLO_ROOT}')
print(f'Train split: images={len(train_images)}, labels={len(train_labels)}')
print(f'Val split: images={len(val_images)}, labels={len(val_labels)}')

## 8. Label Integrity Check

In [ ]:
def validate_label_file(label_path: Path) -> bool:
    lines = [ln for ln in label_path.read_text(encoding='utf-8').splitlines() if ln.strip()]
    for line in lines:
        parts = line.split()
        if len(parts) != 5:
            return False
        values = [float(v) for v in parts[1:]]
        if any(v < 0.0 or v > 1.0 for v in values):
            return False
    return True

sample_labels = train_labels[:200]
invalid_count = 0
for label_file in sample_labels:
    if not validate_label_file(label_file):
        invalid_count += 1

if invalid_count > 0:
    raise RuntimeError(f'Invalid YOLO labels found in sample: {invalid_count}')

print(f'Validated {len(sample_labels)} train labels with YOLO format checks.')

## 9. Visualize Training Samples with Bounding Boxes

In [ ]:
preview_images = train_images[:9]
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for ax, img_path in zip(axes.flatten(), preview_images):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    label_path = TRAIN_LABELS / f'{img_path.stem}.txt'
    if label_path.exists():
        rows = [ln for ln in label_path.read_text(encoding='utf-8').splitlines() if ln.strip()]
        for row in rows:
            cls, cx, cy, bw, bh = row.split()
            cx = float(cx) * w
            cy = float(cy) * h
            bw = float(bw) * w
            bh = float(bh) * h
            x1 = int(cx - bw / 2)
            y1 = int(cy - bh / 2)
            x2 = int(cx + bw / 2)
            y2 = int(cy + bh / 2)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 128), 2)
    ax.imshow(img)
    ax.axis('off')

for ax in axes.flatten()[len(preview_images):]:
    ax.axis('off')

plt.tight_layout()
label_preview_path = ARTIFACTS_DIR / 'sample_training_labels.png'
plt.savefig(label_preview_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved label preview: {label_preview_path}')

## 10. Train YOLO26m Detector

In [ ]:
EPOCHS = 20
IMGSZ = 960
BATCH = 8 if DEVICE == 'cuda' else 4
PRIMARY_MODEL = 'yolo26m.pt'
FALLBACK_MODEL = 'yolo26s.pt'

selected_model = PRIMARY_MODEL
detector = YOLO(selected_model)

try:
    detector.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, project=str(RUNS_DIR), name='number_plate_detection', exist_ok=True, device=0 if DEVICE == 'cuda' else 'cpu', workers=2, seed=SEED, pretrained=True, verbose=True)
except RuntimeError as exc:
    if 'out of memory' not in str(exc).lower():
        raise
    selected_model = FALLBACK_MODEL
    detector = YOLO(selected_model)
    detector.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=max(2, BATCH // 2), project=str(RUNS_DIR), name='number_plate_detection', exist_ok=True, device=0 if DEVICE == 'cuda' else 'cpu', workers=2, seed=SEED, pretrained=True, verbose=True)

run_dir = Path(detector.trainer.save_dir)
best_pt = run_dir / 'weights' / 'best.pt'
if not best_pt.exists():
    raise RuntimeError(f'Missing trained weights: {best_pt}')

print(f'Selected model: {selected_model}')
print(f'Run directory: {run_dir}')

## 11. Evaluate Detection Metrics

In [ ]:
best_detector = YOLO(str(best_pt))
eval_split = 'test' if TEST_IMAGES is not None and TEST_IMAGES.exists() else 'val'
val_result = best_detector.val(data=str(DATA_YAML), split=eval_split, imgsz=IMGSZ, batch=max(1, BATCH // 2), device=0 if DEVICE == 'cuda' else 'cpu', verbose=False)

metrics = {
    'dataset': KAGGLE_DATASET,
    'model': selected_model,
    'epochs': EPOCHS,
    'imgsz': IMGSZ,
    'split': eval_split,
    'precision': float(val_result.box.mp),
    'recall': float(val_result.box.mr),
    'map50': float(val_result.box.map50),
    'map50_95': float(val_result.box.map)
}

metrics_path = ARTIFACTS_DIR / 'metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(json.dumps(metrics, indent=2))

## 12. Optional OCR on Detected Plates (PaddleOCR)

In [ ]:
RUN_OCR = True
OCR_SAMPLE_COUNT = 12

ocr_records = []
preview_frames = []

if RUN_OCR:
    ocr_engine = PaddleOCR(use_angle_cls=True, lang='en', use_gpu=(DEVICE == 'cuda'))
    infer_pool = val_images[:OCR_SAMPLE_COUNT]
    for img_path in infer_pool:
        result = best_detector.predict(source=str(img_path), conf=0.25, verbose=False)[0]
        boxes = result.boxes.xyxy.cpu().numpy() if result.boxes is not None else np.empty((0, 4))
        src = cv2.imread(str(img_path))
        text_list = []
        for box in boxes:
            x1, y1, x2, y2 = [int(v) for v in box]
            crop = src[max(0, y1):max(1, y2), max(0, x1):max(1, x2)]
            if crop.size == 0:
                continue
            ocr_out = ocr_engine.ocr(crop, cls=True)
            if ocr_out and ocr_out[0]:
                parts = [it[1][0] for it in ocr_out[0] if it and len(it) > 1]
                if parts:
                    text_list.append(' '.join(parts))
        ocr_records.append({'image': img_path.name, 'detections': int(len(boxes)), 'texts': text_list})
        preview_frames.append(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))

    ocr_path = ARTIFACTS_DIR / 'ocr_predictions.json'
    ocr_path.write_text(json.dumps(ocr_records, indent=2), encoding='utf-8')

    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    frame_subset = preview_frames[:6]
    for ax, frame in zip(axes.flatten(), frame_subset):
        ax.imshow(frame)
        ax.axis('off')
    for ax in axes.flatten()[len(frame_subset):]:
        ax.axis('off')
    plt.tight_layout()
    ocr_preview = ARTIFACTS_DIR / 'detection_ocr_preview.png'
    plt.savefig(ocr_preview, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved OCR predictions: {ocr_path}')
else:
    print('OCR stage skipped by configuration (RUN_OCR=False).')

## 13. Error Analysis and Limitations

- Motion blur, extreme perspective, and low illumination reduce both detection and OCR quality.
- OCR quality may degrade on region-specific plate styles not represented in the OCR language model.
- If no OCR ground-truth strings are available in the downloaded layout, OCR assessment stays qualitative (saved examples + extracted strings).

## 14. Save Final Artifacts

In [ ]:
onnx_path = Path(best_detector.export(format='onnx'))
final_best_pt = ARTIFACTS_DIR / 'best.pt'
final_best_onnx = ARTIFACTS_DIR / 'best.onnx'
shutil.copy2(best_pt, final_best_pt)
shutil.copy2(onnx_path, final_best_onnx)

manifest = {
    'project': 'Number Plate Detection',
    'dataset': KAGGLE_DATASET,
    'stack': 'YOLO26m + PaddleOCR optional',
    'selected_model': selected_model,
    'artifacts': {
        'weights_pt': str(final_best_pt),
        'weights_onnx': str(final_best_onnx),
        'metrics_json': str(metrics_path),
        'label_preview_png': str(ARTIFACTS_DIR / 'sample_training_labels.png')
    }
}

if RUN_OCR:
    manifest['artifacts']['ocr_predictions_json'] = str(ARTIFACTS_DIR / 'ocr_predictions.json')
    manifest['artifacts']['ocr_preview_png'] = str(ARTIFACTS_DIR / 'detection_ocr_preview.png')

manifest_path = ARTIFACTS_DIR / 'artifact_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))

## 15. Final Summary

This notebook uses real CCPD2019 download, validates YOLO splits/labels, trains and evaluates YOLO26m, optionally runs PaddleOCR on detections, and saves only generated artifacts.